### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
# groq_api_key

In [10]:
from langchain_groq import ChatGroq
model=ChatGroq(model='llama-3.1-8b-instant', api_key=groq_api_key, temperature=0.1)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F7141353C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F716527520>, model_name='llama-3.1-8b-instant', temperature=0.1, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [13]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from French to English"),
    HumanMessage(content="Bonjour Comment allez-vous?")
]

result=model.invoke(messages)
result

AIMessage(content='The translation is:\n\n"Hello, how are you?"\n\n(Note: This is a common greeting in French, and the response is usually "Je vais bien, merci" which translates to "I\'m fine, thank you.")', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 48, 'total_tokens': 94, 'completion_time': 0.091514195, 'completion_tokens_details': None, 'prompt_time': 0.004331003, 'prompt_tokens_details': None, 'queue_time': 0.048405007, 'total_time': 0.095845198}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ff9bd-8920-7750-b8d5-0798907b61b2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 46, 'total_tokens': 94})

In [14]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'The translation is:\n\n"Hello, how are you?"\n\n(Note: This is a common greeting in French, and the response is usually "Je vais bien, merci" which translates to "I\'m fine, thank you.")'

In [15]:
## uSING lcel -CHAIN THE COMPONENTS
chain=model|parser
chain.invoke(messages)

'The translation is:\n\n"Hello, how are you?"'

In [16]:
## pROMPT tEMPLATES
from langchain_core.prompts import ChatPromptTemplate
generic_template='Translate the following into {language}:'
prompt=ChatPromptTemplate.from_messages([
    ('system',generic_template),
    ('user','{text}')
])


In [ ]:
result=prompt.invoke({'language':'French','text':'Hello, how are you?'})

ChatPromptValue(messages=[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})])

In [19]:
chain=prompt|model|parser
chain.invoke({'language':'French','text':'Hello, how are you?'})

'Bonjour, comment allez-vous ?'